# BASELINE 3: AASIST MURNI (STATE-OF-THE-ART BASELINE)
**Arsitektur:** AASIST (Audio Anti-Spoofing using Integrated Spectro-Temporal Graph Attention Networks)  
**Peran:** Sebagai batas atas pembanding (*State-of-the-Art Baseline*). Ini adalah fondasi utama yang akan kita buktikan masih memiliki celah (*bottleneck*), yang nantinya akan kita sempurnakan dengan fusi CQT-LFCC dan Atensi Temporal pada metode ke-4 dan ke-5.

Notebook ini diorganisasikan ke dalam beberapa bagian berikut:
1. **Determinisme & Pengaturan Random Seed:** Mengatur seed acak agar hasil eksperimen dapat diulang secara deterministik.
2. **Persiapan Environment (Clone Repo AASIST):** Mengunduh kode AASIST resmi dan mengatur path agar modul dapat diimpor.
3. **Konfigurasi & Inisiasi Model:** Mendefinisikan parameter konfigurasi AASIST secara eksplisit dan menginisialisasi model.
4. **Tempat Import Dataset & Dataloader (ASVspoof 2019 LA):** Pemuatan dataset lokal dan padding/truncating waveform mentah menjadi tepat 64.600 sampel.
5. **Cross-Dataset Evaluation (ASVspoof 5):** Memuat dataset streaming ASVspoof 5 dari Hugging Face dan mengimplementasikan *Stratified Sampling* untuk mengambil subset berimbang (maksimal 5.000 sampel).
6. **Metrik Evaluasi:** Perhitungan EER dan min t-DCF.
7. **Training & Validation Loop:** Loop pelatihan penuh menggunakan Adam optimizer, CosineAnnealingLR, dan loss function.

## 1. Determinisme & Pengaturan Random Seed
Agar hasil eksperimen bersifat deterministik, kita mengunci seed acak pada Python, NumPy, dan PyTorch.

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed=1234):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(1234)
print("Determinisme aktif. Random seed diatur ke 1234.")

## 2. Persiapan Environment (Clone Repo AASIST)
Untuk menggunakan arsitektur AASIST resmi, kita mengunduh repositori Clova AI AASIST langsung ke dalam *working directory* Kaggle atau lokal. Kita juga menambahkan direktori repositori tersebut ke dalam `sys.path` agar modul model dapat diimpor secara langsung.

In [ ]:
import sys
import subprocess

# Mengunduh repositori AASIST resmi jika belum diunduh
if not os.path.exists('./aasist'):
    print("Mengunduh repositori AASIST dari Clova AI...")
    subprocess.run(["git", "clone", "https://github.com/clovaai/aasist.git"])
else:
    print("Repositori AASIST sudah tersedia.")

# Menambahkan ke sys.path
sys.path.append('./aasist')
print("Path './aasist' berhasil ditambahkan ke sys.path.")

## 3. Konfigurasi & Inisiasi Model
AASIST membutuhkan parameter konfigurasi arsitektur (seperti filter sinc-conv, dimensi GAT, rasio pooling, dan temperatur atensi) untuk inisialisasi model. Kita mendefinisikan parameter konfigurasi default AASIST (sesuai spesifikasi paper Jung et al., 2022) di dalam sebuah dictionary, lalu menginisiasi model `Model` dari `models.AASIST`.

In [ ]:
# Konfigurasi parameter model AASIST
aasist_config = {
    "first_conv": 128,
    "filts": [70, [1, 32], [32, 32], [32, 64], [64, 64]],
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5, 0.5],
    "temperatures": [2.0, 2.0, 100.0, 100.0]
}

try:
    from models.AASIST import Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Model(aasist_config).to(device)
    print(f"Model AASIST berhasil diinisialisasi pada perangkat: {device}")
    
    # Cetak ringkasan singkat parameter model
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameter model: {total_params:,}")
except ImportError as e:
    print(f"Error mengimpor Model AASIST: {e}")
    print("Pastikan folder './aasist' berisi file 'models/AASIST.py' dan path sudah benar.")

## 4. Tempat Import Dataset & Dataloader (ASVspoof 2019 LA)
AASIST menggunakan input berupa sinyal audio mentah (*raw waveform*). Kita mendefinisikan kelas Dataloader kustom untuk memuat file audio, melakukan resampling ke 16kHz jika diperlukan, dan melakukan pemotongan (*truncating*) atau padding dengan nol agar panjang sinyal tepat **64.600 samples** (sesuai standar masukan AASIST).

In [ ]:
# =====================================================================
# TEMPAT UNTUK IMPORT DATASET & INISIALISASI DATALOADER (Mendukung ASVspoof 5 & 2019)
# =====================================================================
from torch.utils.data import Dataset, DataLoader
import torchaudio
import pandas as pd
import numpy as np
import os
import random
import torch

class RawAudioASVDataset(Dataset):
    """
    Dataset PyTorch untuk Audio Anti-Spoofing.
    Mendukung ASVspoof 2019 & ASVspoof 5 dengan pemindaian subfolder secara rekursif.
    Menerapkan Stratified Quota Sampling pada ASVspoof 5 untuk optimasi RAM.
    """
    def __init__(self, protocols_file, wav_dir, max_len=64000, transform=None):
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.transform = transform
        self.file_list = []
        self.labels = []
        
        # Membaca berkas protokol
        if protocols_file and os.path.exists(protocols_file):
            print(f"Membaca protokol dari {protocols_file}...")
            try:
                # Cek baris pertama untuk mendeteksi tipe protokol
                with open(protocols_file, 'r', encoding='utf-8') as f:
                    first_line = f.readline().strip()
                n_cols = len(first_line.split())
                
                if n_cols >= 9: # ASVspoof 5 (10 kolom)
                    print("Mendeteksi format protokol ASVspoof 5. Menerapkan Stratified Quota Sampling...")
                    columns = [
                        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
                        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
                    ]
                    df = pd.read_csv(protocols_file, sep=r'\s+', header=None, names=columns[:n_cols])
                    
                    # Quota sampling (1000 bona fide, dan spoof disampling seimbang per jenis serangan)
                    max_bonafide = 1000
                    df_bonafide = df[df['KEY'] == 'bonafide']
                    df_spoof = df[df['KEY'] == 'spoof']
                    
                    n_bonafide = min(len(df_bonafide), max_bonafide)
                    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
                    
                    df_spoof_sampled = pd.DataFrame()
                    if len(df_spoof) > 0:
                        attack_groups = df_spoof.groupby('ATTACK_LABEL')
                        n_groups = len(attack_groups)
                        samples_per_group = max_bonafide // n_groups
                        
                        for label, group in attack_groups:
                            n_take = min(len(group), samples_per_group)
                            df_grp_sampled = group.sample(n=n_take, random_state=42)
                            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
                            
                        # Penuhi sisa kuota jika ada group yang kurang sampelnya
                        current_collected = len(df_spoof_sampled)
                        if current_collected < max_bonafide:
                            needed = max_bonafide - current_collected
                            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
                            if len(remaining_spoof) > 0:
                                n_take = min(len(remaining_spoof), needed)
                                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                                
                    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
                    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
                    
                    self.file_list = df_final['FLAC_FILE_NAME'].tolist()
                    self.labels = [1 if k == 'bonafide' else 0 for k in df_final['KEY']]
                    print(f"Berhasil memuat {len(self.file_list)} sampel terstratifikasi ASVspoof 5.")
                else: # ASVspoof 2019 (5 kolom)
                    with open(protocols_file, 'r', encoding='utf-8') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                file_name = parts[1]
                                key = parts[4]
                                label = 1 if key == 'bonafide' else 0
                                self.file_list.append(file_name)
                                self.labels.append(label)
                    print(f"Berhasil memuat {len(self.file_list)} entri dataset ASVspoof 2019.")
            except Exception as e:
                print(f"Error membaca berkas protokol: {e}")
                self.file_list = []
                self.labels = []
        else:
            print(f"PERINGATAN: Berkas protokol tidak ditemukan di '{protocols_file}'.")
            print("Membuat DUMMY dataset untuk pengujian...")
            self.file_list = [f"dummy_sample_{i}" for i in range(100)]
            self.labels = [random.choice([0, 1]) for _ in range(100)]

        # Bangun indeks path file audio secara rekursif untuk mendukung folder bersarang (flac_T_aa, dll.)
        self.path_map = {}
        if os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")
        else:
            print(f"PERINGATAN: Direktori audio '{wav_dir}' tidak ditemukan.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        label = self.labels[idx]
        
        # Cari path dari indeks map
        wav_path = self.path_map.get(file_name)
        if not wav_path:
            wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
            if not os.path.exists(wav_path):
                wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
        if wav_path and os.path.exists(wav_path):
            waveform, sr = torchaudio.load(wav_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
            waveform = waveform.squeeze(0)
        else:
            waveform = torch.randn(self.max_len) * 0.01
            
        # Padding atau potong agar ukuran waveform seragam
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        # Terapkan transformasi (ekstraksi fitur) jika disediakan
        if self.transform:
            feature = self.transform(waveform)
            return feature, label
            
        return torch.FloatTensor(waveform), label

# SILAKAN EDIT PATH BASE DIREKTORI DI BAWAH INI SESUAI DENGAN LOKASI DATASET ASVSPOOF 5 LOKAL ANDA
BASE_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled"
PATH_PROTOKOL_TRAIN = os.path.join(BASE_DIR, "ASVspoof5.train.tsv")
PATH_WAV_DIR_TRAIN = BASE_DIR

PATH_PROTOKOL_DEV = os.path.join(BASE_DIR, "ASVspoof5.dev.track_1.tsv")
PATH_WAV_DIR_DEV = BASE_DIR

print("Dataset loader ASVspoof 5 lokal siap digunakan.")


## 5. Cross-Dataset Evaluation (ASVspoof 5) & Stratified Sampling
Untuk mengevaluasi kemampuan generalisasi model AASIST secara objektif pada data lintas domain, kita mengunduh subset dataset **ASVspoof 5** menggunakan pustaka Hugging Face `datasets` secara *streaming*.

Kita membuat fungsi *Stratified Sampling* untuk mengambil maksimal **5.000 sampel** pengujian dengan rasio berimbang 50% *Bona fide* dan 50% *Spoof*, serta membagi porsi kelas spoof secara merata berdasarkan *system ID* (Attack ID) agar evaluasi model tidak bias terhadap jenis serangan tertentu.

In [ ]:
try:
    import datasets
except ImportError:
    print("Menginstall pustaka datasets...")
    subprocess.run(["pip", "install", "datasets", "-q"])
    import datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

def collect_stratified_asvspoof5_tsv(tsv_path, max_samples=5000):
    """
    Membaca berkas protokol TSV ASVspoof 5 secara lokal,
    melakukan stratified sampling:
    - Bona fide: 50%, Spoof: 50%
    - Spoof tersampel secara merata berdasarkan Attack ID (ATTACK_LABEL)
    """
    columns = [
        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
    ]
    print(f"Membaca berkas protokol dari {tsv_path}...")
    df = pd.read_csv(tsv_path, sep=r'\s+', header=None, names=columns)
    print(f"Total baris ditemukan: {len(df)}")
    
    target_class_samples = max_samples // 2
    
    # Pisahkan bona fide dan spoof
    df_bonafide = df[df['KEY'] == 'bonafide']
    df_spoof = df[df['KEY'] == 'spoof']
    
    # Ambil sampel bona fide
    n_bonafide = min(len(df_bonafide), target_class_samples)
    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
    
    # Stratifikasi sampel spoof berdasarkan ATTACK_LABEL
    df_spoof_sampled = pd.DataFrame()
    if len(df_spoof) > 0:
        attack_groups = df_spoof.groupby('ATTACK_LABEL')
        n_groups = len(attack_groups)
        samples_per_group = target_class_samples // n_groups
        
        # Ambil sampel dari setiap group secara merata
        for label, group in attack_groups:
            n_take = min(len(group), samples_per_group)
            df_grp_sampled = group.sample(n=n_take, random_state=42)
            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
            
        current_collected = len(df_spoof_sampled)
        if current_collected < target_class_samples:
            needed = target_class_samples - current_collected
            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
            if len(remaining_spoof) > 0:
                n_take = min(len(remaining_spoof), needed)
                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                
    # Gabungkan
    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    samples_list = []
    for _, row in df_final.iterrows():
        samples_list.append({
            'file_name': row['FLAC_FILE_NAME'],
            'label': 1 if row['KEY'] == 'bonafide' else 0,
            'system_id': row['ATTACK_LABEL']
        })
        
    print(f"Selesai! Mengumpulkan {len(df_bonafide_sampled)} Bona fide dan {len(df_spoof_sampled)} Spoof.")
    return samples_list

def collect_stratified_asvspoof5(dataset_stream, max_samples=5000):
    """
    Mengambil sampel terstratifikasi secara round-robin dari dataset streaming ASVspoof 5 (Hugging Face fallback).
    """
    target_class_samples = max_samples // 2
    bona_fide_samples = []
    spoof_by_attack = {}
    
    print("Mengekstrak sampel dari streaming dataset... Mohon tunggu sebentar.")
    
    for item in dataset_stream:
        label = item.get('label')
        if label is None:
            label = item.get('key') or item.get('class')
            
        is_bonafide = False
        if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
            is_bonafide = True
        elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
            is_bonafide = False
        else:
            is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            
        if is_bonafide:
            if len(bona_fide_samples) < target_class_samples:
                bona_fide_samples.append(item)
        else:
            attack_id = item.get('attack_type') or item.get('attack_id') or item.get('system_id') or item.get('attack') or 'unknown'
            if attack_id not in spoof_by_attack:
                spoof_by_attack[attack_id] = []
            spoof_by_attack[attack_id].append(item)
            
        total_spoof_collected = sum(len(v) for v in spoof_by_attack.values())
        if len(bona_fide_samples) >= target_class_samples and total_spoof_collected >= target_class_samples * 2:
            break
            
    spoof_samples = []
    if spoof_by_attack:
        attack_ids = list(spoof_by_attack.keys())
        idx = 0
        while len(spoof_samples) < target_class_samples:
            added = False
            for aid in attack_ids:
                if idx < len(spoof_by_attack[aid]):
                    spoof_samples.append(spoof_by_attack[aid][idx])
                    added = True
                    if len(spoof_samples) >= target_class_samples:
                        break
            if not added:
                break
            idx += 1
            
    final_samples = bona_fide_samples + spoof_samples
    random.shuffle(final_samples)
    print(f"Selesai! Mengumpulkan {len(bona_fide_samples)} Bona fide dan {len(spoof_samples)} Spoof.")
    return final_samples

# Jalur lokal untuk berkas manifes dan direktori audio
PATH_ASV5_TSV = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
if not os.path.exists(PATH_ASV5_TSV):
    PATH_ASV5_TSV = "../../Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
    
PATH_ASV5_WAV_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/asvspoof5/dev/flac"

try:
    if os.path.exists(PATH_ASV5_TSV):
        print(f"Memuat dataset dari berkas TSV lokal: {PATH_ASV5_TSV}")
        eval_samples = collect_stratified_asvspoof5_tsv(PATH_ASV5_TSV, max_samples=5000)
        use_local_asv5 = True
    else:
        print("Berkas TSV lokal tidak ditemukan. Menggunakan streaming dari Hugging Face...")
        asv5_dataset = load_dataset("jungjee/asvspoof5", streaming=True)
        stream_split = asv5_dataset['train']
        eval_samples = collect_stratified_asvspoof5(stream_split, max_samples=5000)
        use_local_asv5 = False
except Exception as e:
    print(f"Error memuat dataset ASVspoof 5: {e}")
    raise e


## 6. Metrik Evaluasi: Equal Error Rate (EER) & min t-DCF
Evaluasi keandalan sistem audio anti-spoofing diukur menggunakan dua metrik utama:
1. **Equal Error Rate (EER):** Dihitung menggunakan linear interpolation pada DET curve lewat `scipy.optimize.brentq` dan `sklearn.metrics.roc_curve`.
2. **min t-DCF:** Tandem Detection Cost Function untuk menilai dampak performa CM (Countermeasure) jika digabungkan dengan sistem ASV (Automatic Speaker Verification) standar.

In [ ]:
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_eer_scipy(bonafide_scores, spoof_scores):
    """
    Menghitung Equal Error Rate (EER) menggunakan scipy.optimize.brentq dan sklearn.metrics.roc_curve.
    """
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    
    # EER dicapai saat FPR == FNR
    eer = brentq(lambda x: interp1d(fpr, tpr)(x) + x - 1., 0., 1.)
    return eer

def compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores, Pfa_asv=0.22276, Pmiss_asv=0.06677, Pmiss_spoof_asv=0.76843):
    """
    Menghitung minimum Tandem Detection Cost Function (min t-DCF) standar ASVspoof 2019.
    """
    Pspoof = 0.05
    Ptar = (1.0 - Pspoof) * 0.99
    Pnon = (1.0 - Pspoof) * 0.01
    
    Cmiss_cm = 1.0
    Cfa_cm = 10.0
    Cmiss_asv = 1.0
    Cfa_asv = 10.0
    
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    Pmiss_cm = 1.0 - tpr
    Pfa_cm = fpr
    
    C1 = Ptar * (Cmiss_cm - Cmiss_asv * Pmiss_asv) - Pnon * Cfa_asv * Pfa_asv
    C2 = Cfa_cm * Pspoof * (1.0 - Pmiss_spoof_asv)
    
    tDCF = C1 * Pmiss_cm + C2 * Pfa_cm
    tDCF_norm = tDCF / np.minimum(C1, C2)
    min_tdcf = np.min(tDCF_norm)
    return min_tdcf

print("Fungsi EER dan min t-DCF berhasil didefinisikan.")

## 7. Training & Validation Loop
Di bawah ini mendefinisikan fungsi pelatihan untuk model AASIST. Kita menggunakan optimizer `Adam`, scheduler `CosineAnnealingLR` untuk penurunan learning rate secara dinamis, dan `CrossEntropyLoss` sebagai fungsi kerugian.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

def train_aasist_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (AASIST mengembalikan tuple: last_hidden, output)
        _, outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward & optimizer step
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def validate_aasist(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    bonafide_scores = []
    spoof_scores = []
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            _, outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            running_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
            
            # Gunakan probabilitas kelas bona fide (indeks 1) sebagai skor countermeasure
            probs = F.softmax(outputs, dim=1)
            bona_probs = probs[:, 1].cpu().numpy()
            labels_np = batch_y.cpu().numpy()
            
            for score, lbl in zip(bona_probs, labels_np):
                if lbl == 1:
                    bonafide_scores.append(score)
                else:
                    spoof_scores.append(score)
                    
    val_loss = running_loss / total
    val_acc = (correct / total) * 100
    
    bonafide_scores = np.array(bonafide_scores)
    spoof_scores = np.array(spoof_scores)
    
    if len(bonafide_scores) > 0 and len(spoof_scores) > 0:
        val_eer = compute_eer_scipy(bonafide_scores, spoof_scores) * 100
        val_tdcf = compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores)
    else:
        val_eer = 100.0
        val_tdcf = 1.0
        
    return val_loss, val_acc, val_eer, val_tdcf

print("Fungsi pelatihan dan validasi AASIST siap digunakan.")

## 8. Eksekusi Pelatihan & Cross-Dataset Evaluation
Di bawah ini kita menginisialisasi parameter pelatihan, DataLoader untuk ASVspoof 2019 LA, scheduler `CosineAnnealingLR`, dan memulai loop pelatihan baseline AASIST. Setelah pelatihan selesai, kita mengevaluasi model pada dataset lintas-domain **ASVspoof 5** yang telah di-stratifikasi sebelumnya.

In [ ]:
# Parameter Pelatihan
num_epochs = 3  # Silakan sesuaikan dengan kebutuhan eksperimen Anda
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0001)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Pemuatan Dataloader (ASVspoof 2019 LA)
# Jika berkas lokal tidak diatur/ditemukan, kode di bawah akan berjalan dengan dummy dataset.
train_dataset = RawAudioASVDataset(
    protocols_file=PATH_PROTOKOL_TRAIN,
    wav_dir=PATH_WAV_DIR_TRAIN,
    max_len=64600
)

dev_dataset = RawAudioASVDataset(
    protocols_file=PATH_PROTOKOL_DEV,
    wav_dir=PATH_WAV_DIR_DEV,
    max_len=64600
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)

# Loop Pelatihan Baseline AASIST
print(f"\nMemulai Pelatihan AASIST selama {num_epochs} Epoch...\n")
print(f"{'Epoch':^8} | {'Train Loss':^12} | {'Train Acc (%)':^15} | {'Val Loss':^12} | {'Val Acc (%)':^15} | {'Val EER (%)':^15} | {'min t-DCF':^12}")
print("-" * 105)

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_aasist_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device
    )
    
    val_loss, val_acc, val_eer, val_tdcf = validate_aasist(
        model=model,
        dataloader=dev_loader,
        criterion=criterion,
        device=device
    )
    
    scheduler.step()
    
    print(f"{epoch:^8} | {train_loss:^12.4f} | {train_acc:^15.2f} | {val_loss:^12.4f} | {val_acc:^15.2f} | {val_eer:^15.4f} | {val_tdcf:^12.5f}")

print("\nPelatihan selesai!")

class ASVspoof5EvalDataset(Dataset):
    """
    Dataset PyTorch untuk memuat data evaluasi ASVspoof 5 (mendukung mode lokal & streaming HF).
    Mendukung struktur folder bersarang dengan pencarian path secara rekursif.
    """
    def __init__(self, samples, wav_dir, max_len=64600, use_local=True):
        self.samples = samples
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.use_local = use_local
        
        # Bangun indeks path file audio secara rekursif jika menggunakan mode lokal
        self.path_map = {}
        if self.use_local and os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        if self.use_local:
            file_name = item['file_name']
            label = item['label']
            
            # Cari path dari indeks map
            wav_path = self.path_map.get(file_name)
            
            # Jika tidak ditemukan di map, fallback ke os.path.join standar
            if not wav_path:
                wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
                if not os.path.exists(wav_path):
                    wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
            if wav_path and os.path.exists(wav_path):
                waveform, sr = torchaudio.load(wav_path)
                if sr != 16000:
                    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                    waveform = resampler(waveform)
                waveform = waveform.squeeze(0)
            else:
                waveform = torch.randn(self.max_len) * 0.01
        else:
            audio_data = item['audio']['array']
            sr = item['audio']['sampling_rate']
            waveform = torch.tensor(audio_data, dtype=torch.float32)
            
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
                
            if waveform.ndim > 1:
                waveform = waveform.mean(dim=0)
                
            label = item.get('label')
            if label is None:
                label = item.get('key') or item.get('class')
                
            is_bonafide = False
            if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
                is_bonafide = True
            elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
                is_bonafide = False
            else:
                is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            label = 1 if is_bonafide else 0
            
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        return waveform, label
